# CUPED Variance Reduction Analysis
**Project:** Salon Experiment & Causal Analysis – Measuring the True Impact of Business Changes  
**Methodology:** Controlled-experiment Using Pre-Experiment Data (CUPED)  
**Target Metric ($Y$):** Revenue per Visit (INR)  
**Pre-Experiment Covariate ($X$):** Pre-Experiment Baseline Spend (INR)  
**Sample:** 18,000 Enrolled Experiment Units

---

### Ethical Simulation Disclosure
> **NOTICE:** All experimental records in this notebook are synthetically generated for educational demonstration of CUPED variance reduction.


## 1. CUPED Theory & Mathematical Formulation
Controlled-experiment Using Pre-Experiment Data (CUPED, Deng et al., 2013) leverages pre-experiment customer covariates to reduce variance in target experimental metrics without introducing bias.

For target outcome $Y$ and pre-experiment covariate $X$:
$$Y_{i, \text{adj}} = Y_i - \theta (X_i - \bar{X})$$
Where the optimal variance-minimizing coefficient $\theta$ is:
$$\theta = \frac{\widehat{\text{Cov}}(Y, X)}{\widehat{\text{Var}}(X)}$$
The theoretical post-adjustment variance reduction is:
$$\frac{\text{Var}(Y_{\text{adj}})}{\text{Var}(Y)} = 1 - \rho_{XY}^2$$
Where $\rho_{XY}$ is the correlation between $X$ and $Y$.


In [1]:
import os
import sys
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath("../src"))
from causal_inference import compute_cuped_adjustment

DATA_DIR = os.path.abspath("../data/processed")
df_exp = pd.read_csv(os.path.join(DATA_DIR, "experiment_data_clean.csv"))
print("Enrolled sample size:", len(df_exp))
display(df_exp[["customer_id", "treatment_assignment", "pre_experiment_revenue", "revenue_per_visit"]].head())


Enrolled sample size: 18000
  customer_id treatment_assignment  pre_experiment_revenue  revenue_per_visit
0   CUST22554            Treatment                 7399.85            1616.41
1   CUST35432              Control                 7010.55            1357.55
2   CUST16532            Treatment                 5637.93               0.00
3   CUST39462              Control                 1014.84             958.50
4   CUST33478            Treatment                10188.92            1673.51


## 2. CUPED Implementation & Variance Comparison
We apply CUPED on Revenue per Visit ($Y$) using pre-experiment baseline customer spend ($X$) across the full cohort and among customers with completed visits.


In [2]:
# Full Cohort CUPED
cuped_full = compute_cuped_adjustment(
    df=df_exp,
    treatment_col="treatment_assignment",
    outcome_col="revenue_per_visit",
    pre_covariate_col="pre_experiment_revenue"
)

# Active Spending Cohort CUPED
df_active = df_exp[df_exp["revenue_per_visit"] > 0]
cuped_active = compute_cuped_adjustment(
    df=df_active,
    treatment_col="treatment_assignment",
    outcome_col="revenue_per_visit",
    pre_covariate_col="pre_experiment_revenue"
)

cuped_summary = pd.DataFrame([
    {
        "Cohort": "Full Cohort (N = 18,000)",
        "Optimal Theta": cuped_full["theta"],
        "Correlation (rho)": cuped_full["correlation_rho"],
        "Raw Variance": cuped_full["variance_raw"],
        "CUPED Variance": cuped_full["variance_cuped"],
        "Variance Reduction %": f"{cuped_full['actual_variance_reduction_pct']:.2f}%",
        "Raw SE": f"Rs. {cuped_full['raw_se']:.2f}",
        "CUPED SE": f"Rs. {cuped_full['cuped_se']:.2f}",
        "Sample Size Savings %": f"{cuped_full['equivalent_sample_size_savings_pct']:.2f}%"
    },
    {
        "Cohort": "Active Visiting Cohort (N = 10,937)",
        "Optimal Theta": cuped_active["theta"],
        "Correlation (rho)": cuped_active["correlation_rho"],
        "Raw Variance": cuped_active["variance_raw"],
        "CUPED Variance": cuped_active["variance_cuped"],
        "Variance Reduction %": f"{cuped_active['actual_variance_reduction_pct']:.2f}%",
        "Raw SE": f"Rs. {cuped_active['raw_se']:.2f}",
        "CUPED SE": f"Rs. {cuped_active['cuped_se']:.2f}",
        "Sample Size Savings %": f"{cuped_active['equivalent_sample_size_savings_pct']:.2f}%"
    }
])
display(cuped_summary)


                                Cohort  Optimal Theta  Correlation (rho)  Raw Variance  CUPED Variance Variance Reduction %     Raw SE  CUPED SE Sample Size Savings %
0             Full Cohort (N = 18,000)         0.0332             0.1961     456305.15       438757.95                3.85%  Rs. 10.01  Rs. 9.81                 3.85%
1  Active Visiting Cohort (N = 10,937)         0.0308             0.3629     114566.70        99474.89               13.17%   Rs. 6.10  Rs. 5.67                13.17%


## 3. Unbiasedness & Sensitivity Validation
Notice that the estimated treatment effect point estimate remains essentially unchanged between raw and adjusted models, but the standard error shrinks noticeably, tightening the 95% confidence intervals.


In [3]:
precision_comparison = pd.DataFrame([
    {
        "Model": "Standard OLS (Raw)",
        "Estimated Effect": f"Rs. {cuped_full['raw_treatment_effect']:+.2f}",
        "Standard Error": f"Rs. {cuped_full['raw_se']:.3f}",
        "95% CI": f"[Rs. {cuped_full['raw_ci_95'][0]:+.2f}, Rs. {cuped_full['raw_ci_95'][1]:+.2f}]",
        "p-value": f"{cuped_full['raw_p_value']:.4e}"
    },
    {
        "Model": "CUPED Adjusted",
        "Estimated Effect": f"Rs. {cuped_full['cuped_treatment_effect']:+.2f}",
        "Standard Error": f"Rs. {cuped_full['cuped_se']:.3f}",
        "95% CI": f"[Rs. {cuped_full['cuped_ci_95'][0]:+.2f}, Rs. {cuped_full['cuped_ci_95'][1]:+.2f}]",
        "p-value": f"{cuped_full['cuped_p_value']:.4e}"
    }
])
display(precision_comparison)


                Model Estimated Effect Standard Error                      95% CI     p-value
0  Standard OLS (Raw)      Rs. +145.27     Rs. 10.012  [Rs. +125.65, Rs. +164.89]  1.9478e-47
1      CUPED Adjusted      Rs. +146.27      Rs. 9.814  [Rs. +127.03, Rs. +165.50]  6.2244e-50


## 4. Key Takeaways & Experimentation Efficiency
1. **Unbiased Point Estimates:** CUPED preserves point-estimate unbiasedness while systematically reducing standard errors by removing pre-treatment historical variance.
2. **Business Value of Variance Reduction:** Achieving a ~13.2% variance reduction in active spend means an A/B experiment requires 13.2% fewer sample units or can reach conclusive statistical power 13.2% faster, accelerating business iteration cycles.
3. **No Look-Ahead Contamination:** Crucially, covariate $X$ was measured strictly before the experiment start date (`2024-07-01`), avoiding post-treatment bias.
